In [ ]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-3.5-turbo-0125')

In [ ]:
def get_response(prompt):
    response = model(prompt)
    return response
def main():
    prompt = "What is the capital of France?"
    response = get_response(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
if __name__ == "__main__":  
    main()
# This script demonstrates how to use the ChatOpenAI class from the langchain_openai module.
# It loads environment variables, initializes a ChatOpenAI model with a specific version,
# defines a function to get a response from the model based on a prompt, and prints the prompt and response.
# The script is designed to be run as a standalone program, and it will print the response to the console.
# The script uses the dotenv library to load environment variables from a .env file,
# which is useful for managing sensitive information like API keys.
# The script is structured with a main function that serves as the entry point,
# ensuring that the code runs only when the script is executed directly.
# The use of the __name__ == "__main__" construct is a common Python practice
# to allow or prevent parts of code from being run when the modules are imported.
# The script is simple and straightforward, making it easy to understand and modify.
# The script can be extended or modified to include more complex functionality,
# such as handling different types of prompts or integrating with other systems.    
# The script is designed to be easily adaptable for various use cases,
# allowing developers to customize it according to their needs.
# The script can be used as a starting point for building more complex applications
# that leverage the capabilities of the ChatOpenAI model.
# The script is a good example of how to interact with the OpenAI API

CRIANDO CLASSES

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
from langchain_openai import ChatOpenAI

class pydPessoa(BaseModel):
    nome: str
    idade: int
    email: Optional[str] = None
    telefone: Optional[str] = None
    endereco: Optional[str] = None

danilo = pydPessoa(nome ='Danilo', idade = '57', 
                       email = 'danilopor@gmail.com', 
                       telefone = '81 994728217', 
                       endereco= 'Rua das Flores, 123'
                       )

AnaJulia = pydPessoa(nome ='Ana Julia', 
                     idade = '50',
                     telefone= '81 994728217',
                     endereco= 'Rua das Flores, 123'
                     )

# NESTING para criar um objeto dentro de outro

class pydPessoa(BaseModel):
    funcionarios:list[pydPessoa]
    def __init__(self, funcionarios: List[pydPessoa]):
        self.funcionarios = funcionarios

MESMA COISA USANDO LANGCHAIN

In [ ]:
class pydEmpresa(BaseModel):
    nome: str
    cpf: str
    funcionarios: List[pydPessoa]
    endereco: Optional[str] = None
    telefone: Optional[str] = None
    email: Optional[str] = None
    def __init__(self, nome: str, 
                 cpf: str, 
                 funcionarios: List[pydPessoa], 
                 endereco: Optional[str] = None, 
                 telefone: Optional[str] = None, 
                 email: Optional[str] = None):
        self.nome = nome
        self.cpf = cpf
        self.funcionarios = funcionarios
        self.endereco = endereco
        self.telefone = telefone
        self.email = email


CRIANDO UM AGENTE

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import chat

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Você é um Redator Publicitário experiente, criativo e genial. Sua habilidade é escrever textos criativos e persuasivos com clareza, gramática correta e estilo adaptável a diferentes públicos e marcas para diferentes formatos. Criar mensagens que motivem a ação do público-alvo, com conhecimento sobre como influenciar e persuadir um público, através de parcerias com outras marcas ou influenciadores e capacidade de contar histórias envolventes.'),
    ('user', '{input}'),
    ])

chain = prompt | chat.bind(model)
def get_response(prompt):
    response = chain.invoke({"input": prompt})
    return response
def main():
    prompt = "Crie um texto criativo e persuasivo para uma campanha publicitária de uma nova linha de produtos de beleza, destacando os benefícios e diferenciais da marca."
    response = get_response(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
if __name__ == "__main__":
    main()


Função para obter emails

In [ ]:
import imaplib
import email
from email.header import decode_header
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

class TipoEmail(str, Enum):
    todos = 'todos'
    não_lidos = 'não lidos'
    lidos = 'lidos'

class ObterEmails(BaseModel):
    """Obtém os emails de acordo com o tipo e a quantidade desejada."""
    tipo: TipoEmail = Field(description="Tipo de emails a serem obtidos.")
    quantidade: Optional[int] = Field(default=5, description="Quantidade de emails a serem obtidos. Padrão: 5")

def obter_emails(email_usuario, senha, tipo="todos", quantidade=5):
    try:
        # Conectar ao servidor IMAP (Exemplo para Gmail)
        imap = imaplib.IMAP4_SSL("imap.gmail.com")

        # Autenticar
        imap.login(email_usuario, senha)

        # Selecionar a caixa de entrada
        imap.select("inbox")

        # Definir filtro com base no tipo escolhido
        filtro = "ALL" if tipo == "todos" else "UNSEEN" if tipo == "não lidos" else "SEEN"
        
        # Buscar emails de acordo com o filtro
        status, mensagens = imap.search(None, filtro)

        # Pegar os últimos X emails
        email_ids = mensagens[0].split()[-quantidade:]

        emails = []
        for email_id in email_ids:
            status, msg_data = imap.fetch(email_id, "(RFC822)")
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    assunto, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(assunto, bytes):
                        assunto = assunto.decode(encoding or "utf-8")
                    emails.append({"Assunto": assunto, "De": msg["From"]})

        # Fechar conexão
        imap.close()
        imap.logout()

        return emails

    except Exception as e:
        return f"Erro ao obter emails: {e}"

# Exemplo de uso
email_usuario = "danilopor@gmail.com"
senha = "@s"  #  Melhor usar autenticação por app password
print(obter_emails(email_usuario, senha, tipo="não lidos", quantidade=3))

ANALISE DE SENTIMENTO NUM TEXTO
Ver se as mensagens que o cliente recebe são positivas ou negativas

In [ ]:
from pydantic import BaseModel, Field #Importação atualizada
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

class Sentimento(BaseModel):
    '''Define o sentimento e a língua da mensagem enviada'''
    sentimento: str = Field(description='Sentimendo do texto. Deve ser "pos", "neg" ou "nd" para não definido.')
    lingua: str = Field(description='Língua que o texto foi escrito (deve estar no formato ISO 639-1)')

tool_sentimento = convert_to_openai_function(Sentimento)
tool_sentimento

texto = 'Eu gosto muito de massa aos quatro queijos'

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Pense com cuidado ao categorizar o texto conforme as instruções'),
    ('user', '{input}')
])
chat = ChatOpenAI()

chain = (prompt 
         | chat.bind(functions=[tool_sentimento], function_call={'name': 'Sentimento'})
         | JsonOutputFunctionsParser())

chain.invoke(texto)

## ENCAMINHANDO EMAIL PARA SETORES DIFERENTES

Digamos que temos um chatbot e queremos fazer um roteamento para os setores de interesse, no nosso caso atendimento_cliente, duvidas_alunos, vendas, spam. A primeira etapa é criar um modelo que entenda a solicitação do cliente e direcione para o setor certo. Nas próximas etapas o atendimento pode ser continuado por pessoas ou por agentes especializados para as tarefas do setor.
Retiramos as seguintes mensagens de email recebidos pela nossa equipe de atendimento e vamos tentar criar um direcionamento para elas:

In [ ]:
duvida = [
    'Bom dia, gostaria de saber se há um certificado final para cada trilha ou se os certificados são somente para os cursos e projetos? Obrigado!',
    'In Etsy, Amazon, eBay, Shopify https://pint77.com Pinterest+SEO +II = high sales results',
    'Boa tarde, estou iniciando hoje e estou perdido. Tenho vários objetivos. Não sei nada programação, exceto que utilizo o Power automate desktop da Microsoft. Quero aprender tudo na plataforma que se relacione ao Trading de criptomoedas. Quero automatizar Tradings, fazer o sistema reconhecer padrões, comprar e vender segundo critérios que eu defina, etc. Também tenho objetivos de aprender o máximo para utilizar em automações no trabalho também, que envolve a área jurídica e trabalho em processos. Como sou fã de eletrônica e tenho cursos na área, também queria aprender o que precisa para automatizacões diversas. Existe algum curso ou trilha que me prepare com base para todas essas áreas ao mesmo tempo e a partir dele eu aprenda isoladamente aquilo que seria exigido para aplicar aos meus projetos?',
    'Bom dia, Havia pedido cancelamento de minha mensalidade no mes 2 e continuaram cobrando. Peço cancelamento da assinatura. Peço por gentileza, para efetivarem o cancelamento da assomatura e pagamento.',
    'Bom dia. Não estou conseguindo tirar os certificados dos cursos que concluí. Por exemplo, já consegui 100% no python starter, porém, não consigo tirar o certificado. Como faço?',
    'Bom dia. Não enconte no site o preço de um curso avulso. SAberiam me informar?'
    ]

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field #Importação atualizada

class SetorEnum(str, Enum):
    atendimento_cliente = 'atendimento_cliente'
    duvidas_aluno = 'duvidas_aluno'
    vendas = 'vendas'
    spam = 'spam'

class DirecionaSetorResponsavel(BaseModel):
    """Direciona a dúvida de um cliente ou aluno da escola de programação Asimov para o setor responsável"""
    setor: SetorEnum

tool_direcionamento = convert_to_openai_function(DirecionaSetorResponsavel)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser
from langchain_core.utils.function_calling import convert_to_openai_function

system_message = '''Pense com cuidado ao categorizar o texto conforme as instruções.
Questões relacionadas a dúvidas de preço, sobre o produto, como funciona devem ser direciodas para "vendas".
Questões relacionadas a conta, acesso a plataforma, a cancelamento e renovação de assinatura para devem ser direciodas para "atendimento_cliente".
Questões relacionadas a dúvidas técnicas de programação, conteúdos da plataforma ou tecnologias na área da programação devem ser direciodas para "duvidas_alunos".
Mensagens suspeitas, em outras línguas que não português, contendo links devem ser direciodas para "spam".
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_message),
    ('user', '{input}')
])
chat = ChatOpenAI()

chain = (prompt | 
chat.bind(functions=[tool_direcionamento], function_call={'name': 'DirecionaSetorResponsavel'}) 
| JsonOutputFunctionsParser())


In [ ]:
duvida = duvida[0]
resposta = chain.invoke({'input': duvida})

print("Dúvida: " ,duvida)
print("Setor: " ,resposta['setor'])
print("Resposta: " ,resposta)

Extraindo informações de um texto

In [ ]:
texto = '''A Apple foi fundada em 1 de abril de 1976 por Steve Wozniak, Steve Jobs e Ronald Wayne 
com o nome de Apple Computers, na Califórnia. O nome foi escolhido por Jobs após a visita do pomar 
de maçãs da fazenda de Robert Friedland, também pelo fato do nome soar bem e ficar antes da Atari 
nas listas telefônicas.

O primeiro protótipo da empresa foi o Apple I que foi demonstrado na Homebrew Computer Club em 1975, 
as vendas começaram em julho de 1976 com o preço de US$ 666,66, aproximadamente 200 unidades foram 
vendidas,[21] em 1977 a empresa conseguiu o aporte de Mike Markkula e um empréstimo do Bank of America.'''

In [ ]:
from pydantic import BaseModel, Field #Importação atualizada
from typing import List
from langchain_core.utils.function_calling import convert_to_openai_function

class Acontecimento(BaseModel):
    '''Informação sobre um acontecimento'''
    data: str = Field(description='Data do acontecimento no formato YYYY-MM-DD')
    acontecimento: str = Field(description='Acontecimento extraído do texto')

class ListaAcontecimentos(BaseModel):
    """Acontecimentos para extração"""
    acontecimentos: List[Acontecimento] = Field(description='Lista de acontecimentos presentes no texto informado')

tool_acontecimentos = convert_to_openai_function(ListaAcontecimentos)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extraia as frases de acontecimentos. Elas devem ser extraídas integralmente'),
    ('user', '{input}')
])
chat = ChatOpenAI()

chain = (prompt 
         | chat.bind(functions=[tool_acontecimentos], function_call={'name': 'ListaAcontecimentos'}))

from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

chain = (prompt 
         | chat.bind(functions=[tool_acontecimentos], function_call={'name': 'ListaAcontecimentos'})
         | JsonKeyOutputFunctionsParser(key_name='acontecimentos'))

chain.invoke({'input': texto})

Extraindo Informações da WEB

In [ ]:
from langchain_community.document_loaders.web_base import WebBaseLoader

loader = WebBaseLoader('https://hub.asimov.academy/blog/')
page = loader.load()


In [ ]:
from pydantic import BaseModel, Field #Importação atualizada
from typing import List
from langchain_core.utils.function_calling import convert_to_openai_function

class BlogPost(BaseModel):
    '''Informações sobre um post de blog'''
    titulo: str = Field(description='O título do post de blog')
    autor: str = Field(description='O autor do post de blog')
    data: str = Field(description='A data de publicação do post de blog')

class BlogSite(BaseModel):
    '''Lista de blog posts de um site'''
    posts: List[BlogPost] = Field(description='Lista de posts de blog do site')

tool_blog = convert_to_openai_function(BlogSite)


In [ ]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extraia da página todos os posts de blog com autor e data de publicação'),
    ('user', '{input}')
])
chat = ChatOpenAI()
chain = (prompt 
         | chat.bind(functions=[tool_blog], function_call={'name': 'BlogSite'})
         | JsonKeyOutputFunctionsParser(key_name='posts'))

chain.invoke({'input': page})

Exercicio: Extrair ingredientes de uma receita para um AGENTE que faz compras automaticamente.

In [ ]:
import csv
from pydantic import BaseModel, Field 
from typing import List
from langchain_core.utils.function_calling import convert_to_openai_function
from enum import Enum
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

class TipoEnum(str, Enum):
    utensilio = 'utensilio'
    ingrediente = 'ingrediente'

class Item(BaseModel):
    '''item e utensilios usados na receita'''
    tipo: TipoEnum = Field(description='Tipo do item usado na receita')
    descricao: str = Field(description='Descrição do item usados na receita')

class listaItens(BaseModel):
    """ingredientes da receita"""
    lista: List[Item] = Field(description='Lista de ingredientes e utencilios presentes na receita')

tool_receita = convert_to_openai_function(listaItens)

system_message = '''Você é um assistente de receitas. Sua tarefa é extrair os ingredientes e utensílios usados na receita. A receita pode ter ingredientes e utensílios repetidos.'''
prompt = ChatPromptTemplate.from_messages([
    ('system', system_message),
    ('user', '{input}')
])
chat = ChatOpenAI()

chain = (prompt 
         | chat.bind(functions=[tool_receita], function_call={'name': 'listaItens'})
         | JsonKeyOutputFunctionsParser(key_name='lista'))

bolo = ''' Massa
Em um liquidificador, adicione a cenoura, os ovos e o óleo, depois misture.
Acrescente o açúcar e bata novamente por 5 minutos.
Em uma tigela ou na batedeira, adicione a farinha de trigo e depois misture novamente.
Acrescente o fermento e misture lentamente com uma colher.
Asse em um forno preaquecido a 180° C por aproximadamente 40 minutos.
Cobertura
Despeje em uma tigela a manteiga, o chocolate em pó, o açúcar e o leite, depois misture.
Leve a mistura ao fogo e continue misturando até obter uma consistência cremosa, depois despeje a calda por cima do bolo.'''

resposta = chain.invoke({'input': bolo})

ingredientes = [item for item in resposta if item['tipo'] == 'ingrediente']
utensilios = [item for item in resposta if item['tipo'] == 'utensilio']

print('Ingredientes:', ingredientes)
print('Utensilios:', utensilios)

def salvar_csv(lista, nome_arquivo):
    with open(nome_arquivo, mode='w', newline='', encoding='utf-8') as arquivo:
        writer = csv.DictWriter(arquivo, fieldnames=['descricao'])
        writer.writeheader()
        for item in lista:
            writer.writerow({'descricao': item['descricao']})

salvar_csv(ingredientes, '/Users/daniloportela/Desktop/MATEUS/ingredientes.csv')
salvar_csv(utensilios, '/Users/daniloportela/Desktop/MATEUS/utensilios.csv')            

Enviando EMAILS

In [ ]:
import os
import ssl
import smtplib
from langchain.agents import tool
from pydantic import BaseModel, Field
from email.message import EmailMessage
from langchain_openai import ChatOpenAI
from dotenv import find_dotenv, load_dotenv
from langchain.agents.agent import AgentFinish
from langchain.prompts import ChatPromptTemplate
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser


_ = load_dotenv(find_dotenv())
GMAIL_SENHA_APP = os.environ["@serominha68"]
EMAIL_USUARIO = os.environ["danilopor@gmail.com"]

class RetornaTempArgs(BaseModel):
    destinatario: str = Field(description="Email do destinatário da mensagem")
    titulo: str = Field(description="Título do email à ser enviado")
    corpo: str = Field(description="Mensagem do email à ser enviado")

@tool(args_schema=RetornaTempArgs)
def envia_email(destinatario, titulo, corpo):
    '''Função de envio de email'''
    email_usuario = EMAIL_USUARIO
    senha_app = GMAIL_SENHA_APP
    message_email = EmailMessage()
    message_email['From'] = email_usuario
    message_email['To'] = destinatario
    message_email['Subject'] = titulo

    message_email.set_content(corpo)
    safe = ssl.create_default_context()

    with smtplib.SMTP_SSL('smtp.gmail.com', 465, context=safe) as smtp:
        smtp.login(email_usuario, senha_app)
        smtp.sendmail(email_usuario, destinatario, message_email.as_string())

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Você é um asistente amigável de envio de email chamado GenBot'),
    ('user', '{input}')
])

def roteamento(response , tools):
    tool_run = {tool.name: tool for tool in tools}
    if isinstance(response, AgentFinish):
        return response.return_values['output']
    else:
        return tool_run[response.tool].run(response.tool_input)
    
chat = ChatOpenAI()
tools = [envia_email]
tools_json = [convert_to_openai_function(tool) for tool in tools]

chain = prompt | chat.bind(functions=tools_json, function_call={'name': 'envia_email'}) | OpenAIFunctionsAgentOutputParser() | (lambda response: roteamento(response, tools))

response = chain.invoke({'input': '{"destinatario": "seu_email@hotmail.com", "titulo":"Utilização de tools", "assunto":"Isso é um email de teste"}'}) 
